# 08 — Hyperparameter experiments

**Project:** Hotel overbooking decision support  
**Stage:** Controlled model selection using Train only  
**Purpose:** Test a small number of model settings without using the Choose or Proof outcomes to tune them.

A hyperparameter is a setting chosen before a model is fitted. Examples include the regularisation strength in logistic regression and the minimum number of rows required in a random-forest leaf.

This notebook:
1. Creates chronological, expanding validation folds inside Train.
2. Tests a small, pre-declared set of settings.
3. Selects settings using the average validation Brier score.
4. Fits the selected model on all Train rows.
5. Evaluates the frozen result on Choose once.

**Do not use Proof data here.** The Proof period is reserved for the final evaluation.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Load data and create the Train and Choose periods

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src import data, contract

raw = data.read_raw()
bookings = raw.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = (
    bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
)

train, choose, proof = contract.split_periods(bookings)

print("Train rows:", f"{len(train):,}")
print("Choose rows:", f"{len(choose):,}")
print("Proof rows:", f"{len(proof):,}")

Train rows: 49,157
Choose rows: 29,546
Proof rows: 40,687


## 2. Prepare the same candidate features as Notebook 07

We deliberately hold `country` back, as in Notebooks 04–07. The current `src/features.py` still differs and must be reconciled before the final reusable pipeline is approved.

In [3]:
categorical_columns = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company"
]

numeric_columns = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

feature_columns = categorical_columns + numeric_columns

def prepare_features(frame):
    X = frame[feature_columns].copy()

    for column in categorical_columns:
        X[column] = X[column].astype("string")

    X["agent"] = X["agent"].fillna("NO_AGENT")
    X["company"] = X["company"].fillna("NO_COMPANY")

    for column in ["hotel", "market_segment", "distribution_channel"]:
        X[column] = X[column].fillna("UNKNOWN_" + column.upper())

    return X

X_train = prepare_features(train)
y_train = train["is_canceled"].astype(int)

X_choose = prepare_features(choose)
y_choose = choose["is_canceled"].astype(int)

print("Candidate feature columns:", feature_columns)

Candidate feature columns: ['hotel', 'market_segment', 'distribution_channel', 'agent', 'company', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']


## 3. Create chronological expanding folds within Train

Random cross-validation would mix earlier and later bookings. Instead, we:
- sort Train by arrival date;
- divide its unique arrival dates into four consecutive blocks;
- use block 1 to train and block 2 to validate;
- use blocks 1–2 to train and block 3 to validate;
- use blocks 1–3 to train and block 4 to validate.

The split is based on unique dates, so bookings from the same arrival date are not split across training and validation in a fold.

In [4]:
ordered_dates = np.array(sorted(train["arrival_date"].dropna().unique()))
date_blocks = np.array_split(ordered_dates, 4)

folds = []

for fold_number in range(1, 4):
    training_dates = np.concatenate(date_blocks[:fold_number])
    validation_dates = date_blocks[fold_number]

    train_mask = train["arrival_date"].isin(training_dates)
    validation_mask = train["arrival_date"].isin(validation_dates)

    train_index = np.flatnonzero(train_mask.to_numpy())
    validation_index = np.flatnonzero(validation_mask.to_numpy())

    folds.append((train_index, validation_index))

    print(
        f"Fold {fold_number}:",
        f"train {train.loc[train_mask, 'arrival_date'].min().date()} to "
        f"{train.loc[train_mask, 'arrival_date'].max().date()} |",
        f"validate {train.loc[validation_mask, 'arrival_date'].min().date()} to "
        f"{train.loc[validation_mask, 'arrival_date'].max().date()} |",
        f"{len(train_index):,} train rows, {len(validation_index):,} validation rows"
    )

    assert (
        train.loc[train_mask, "arrival_date"].max()
        < train.loc[validation_mask, "arrival_date"].min()
    )

Fold 1: train 2015-07-01 to 2015-09-30 | validate 2015-10-01 to 2015-12-31 | 11,779 train rows, 10,217 validation rows
Fold 2: train 2015-07-01 to 2015-12-31 | validate 2016-01-01 to 2016-03-31 | 21,996 train rows, 10,963 validation rows
Fold 3: train 2015-07-01 to 2016-03-31 | validate 2016-04-01 to 2016-06-30 | 32,959 train rows, 16,198 validation rows


## 4. Define preprocessing inside each model pipeline

The one-hot encoder and numeric imputer/scaler are fitted separately inside each fold, using that fold's training rows only. This avoids learning category information or numeric statistics from its validation block.

In [5]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_columns
            ),
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_columns
            )
        ],
        remainder="drop"
    )

## 5. Declare a small hyperparameter search

We test:
- Logistic regression with three regularisation strengths (`C`).
- Random forest with three combinations of tree count and minimum leaf size.

This is intentionally a small search. A larger search costs more time and increases the risk of selecting settings that fit quirks in this historical sample.

In [6]:
candidate_settings = [
    {
        "model": "logistic_regression",
        "settings": {"C": 0.1}
    },
    {
        "model": "logistic_regression",
        "settings": {"C": 1.0}
    },
    {
        "model": "logistic_regression",
        "settings": {"C": 10.0}
    },
    {
        "model": "random_forest",
        "settings": {"n_estimators": 100, "min_samples_leaf": 10}
    },
    {
        "model": "random_forest",
        "settings": {"n_estimators": 150, "min_samples_leaf": 20}
    },
    {
        "model": "random_forest",
        "settings": {"n_estimators": 150, "min_samples_leaf": 40}
    }
]

display(pd.DataFrame(candidate_settings))

,model,settings
0,logistic_regression,{'C': 0.1}
1,logistic_regression,{'C': 1.0}
2,logistic_regression,{'C': 10.0}
3,random_forest,"{'n_estimators': 100, 'min_samples_leaf': 10}"
4,random_forest,"{'n_estimators': 150, 'min_samples_leaf': 20}"
5,random_forest,"{'n_estimators': 150, 'min_samples_leaf': 40}"


## 6. Create a model from a setting

Each candidate gets a fresh preprocessor. No fitted transformation is shared between folds.

In [7]:
def make_model(candidate):
    if candidate["model"] == "logistic_regression":
        estimator = LogisticRegression(
            C=candidate["settings"]["C"],
            max_iter=1000
        )
    elif candidate["model"] == "random_forest":
        estimator = RandomForestClassifier(
            n_estimators=candidate["settings"]["n_estimators"],
            min_samples_leaf=candidate["settings"]["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )
    else:
        raise ValueError("Unknown model type")

    return Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", estimator)
    ])

## 7. Run the chronological validation experiment

For every candidate and fold:
1. Fit on the earlier block or blocks.
2. Predict cancellation probabilities on the next block.
3. Calculate Brier score and log loss.

The model-selection metric is the mean Brier score across the three folds. Log loss is retained as a secondary diagnostic.

In [8]:
validation_rows = []

for candidate_number, candidate in enumerate(candidate_settings, start=1):
    for fold_number, (fit_index, validation_index) in enumerate(folds, start=1):
        model = make_model(candidate)

        model.fit(
            X_train.iloc[fit_index],
            y_train.iloc[fit_index]
        )

        classes = list(model.named_steps["model"].classes_)
        positive_position = classes.index(1)

        probabilities = model.predict_proba(
            X_train.iloc[validation_index]
        )[:, positive_position]

        actual = y_train.iloc[validation_index]

        validation_rows.append({
            "candidate_number": candidate_number,
            "model": candidate["model"],
            "settings": str(candidate["settings"]),
            "fold": fold_number,
            "validation_rows": len(validation_index),
            "brier_score": brier_score_loss(actual, probabilities),
            "log_loss": log_loss(actual, probabilities, labels=[0, 1])
        })

validation_results = pd.DataFrame(validation_rows)
display(validation_results)

,candidate_number,model,settings,fold,validation_rows,brier_score,log_loss
0,1,logistic_regression,{'C': 0.1},1,10217,0.128264,0.413856
1,1,logistic_regression,{'C': 0.1},2,10963,0.168577,0.581398
2,1,logistic_regression,{'C': 0.1},3,16198,0.222320,0.642072
3,2,logistic_regression,{'C': 1.0},1,10217,0.127891,0.404536
4,2,logistic_regression,{'C': 1.0},2,10963,0.172162,0.607275
5,2,logistic_regression,{'C': 1.0},3,16198,0.227383,0.665588
6,3,logistic_regression,{'C': 10.0},1,10217,0.135103,0.428449
7,3,logistic_regression,{'C': 10.0},2,10963,0.174734,0.631387
8,3,logistic_regression,{'C': 10.0},3,16198,0.230832,0.685472
9,4,random_forest,"{'n_estimators': 100, 'min_samples_leaf': 10}",1,10217,0.143289,0.456945


## 8. Summarise candidates and select settings

The candidate with the lowest average Brier score is selected. We also inspect variation across folds: a small average advantage with unstable fold results may not be meaningful.

In [9]:
candidate_summary = (
    validation_results.groupby(
        ["candidate_number", "model", "settings"],
        as_index=False
    )
    .agg(
        mean_brier_score=("brier_score", "mean"),
        std_brier_score=("brier_score", "std"),
        mean_log_loss=("log_loss", "mean")
    )
    .sort_values("mean_brier_score")
    .reset_index(drop=True)
)

display(candidate_summary)

selected_candidate_number = int(
    candidate_summary.iloc[0]["candidate_number"]
)

selected_candidate = candidate_settings[selected_candidate_number - 1]

print("Selected model:", selected_candidate["model"])
print("Selected settings:", selected_candidate["settings"])
print("Selection basis: lowest mean chronological validation Brier score")

,candidate_number,model,settings,mean_brier_score,std_brier_score,mean_log_loss
0,1,logistic_regression,{'C': 0.1},0.173054,0.047188,0.545775
1,2,logistic_regression,{'C': 1.0},0.175812,0.049846,0.559133
2,4,random_forest,"{'n_estimators': 100, 'min_samples_leaf': 10}",0.176018,0.036137,0.529855
3,5,random_forest,"{'n_estimators': 150, 'min_samples_leaf': 20}",0.178427,0.032034,0.537579
4,3,logistic_regression,{'C': 10.0},0.180223,0.048100,0.581769
5,6,random_forest,"{'n_estimators': 150, 'min_samples_leaf': 40}",0.182811,0.029016,0.548927


Selected model: logistic_regression
Selected settings: {'C': 0.1}
Selection basis: lowest mean chronological validation Brier score


## 9. Fit the selected settings on all Train rows

After selection, we refit the selected configuration on the complete Train period. The resulting pipeline is then evaluated on Choose.

In [10]:
selected_model = make_model(selected_candidate)
selected_model.fit(X_train, y_train)

selected_classes = list(selected_model.named_steps["model"].classes_)
selected_positive_position = selected_classes.index(1)

choose_probability = selected_model.predict_proba(
    X_choose
)[:, selected_positive_position]

choose_metrics = {
    "model": selected_candidate["model"],
    "settings": str(selected_candidate["settings"]),
    "choose_brier_score": brier_score_loss(y_choose, choose_probability),
    "choose_log_loss": log_loss(y_choose, choose_probability, labels=[0, 1])
}

display(pd.DataFrame([choose_metrics]))

,model,settings,choose_brier_score,choose_log_loss
0,logistic_regression,{'C': 0.1},0.209066,0.610456


## 10. Compare the tuned model with the Notebook 07 defaults

This is a descriptive comparison on Choose. Do not go back and repeatedly adjust settings after seeing this result. If the result is disappointing, record it and define a new experiment before running it.

In [11]:
default_models = {
    "logistic_regression_default": Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "random_forest_default": Pipeline(steps=[
        ("preprocess", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

comparison_rows = [{
    "model": "selected_tuned_model",
    "brier_score": brier_score_loss(y_choose, choose_probability),
    "log_loss": log_loss(y_choose, choose_probability, labels=[0, 1])
}]

for model_name, model in default_models.items():
    model.fit(X_train, y_train)
    classes = list(model.named_steps["model"].classes_)
    positive_position = classes.index(1)
    probabilities = model.predict_proba(X_choose)[:, positive_position]

    comparison_rows.append({
        "model": model_name,
        "brier_score": brier_score_loss(y_choose, probabilities),
        "log_loss": log_loss(y_choose, probabilities, labels=[0, 1])
    })

choose_comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("brier_score")
    .reset_index(drop=True)
)

display(choose_comparison)

,model,brier_score,log_loss
0,selected_tuned_model,0.209066,0.610456
1,random_forest_default,0.210250,0.607264
2,logistic_regression_default,0.210995,0.617134


## 11. Save the experiment record

We save the fold-level results, candidate summary, selected settings and Choose comparison. The selected model is not yet treated as the final calibrated or approved model.

In [12]:
evidence_folder = Path("evidence/hyperparameter_experiments")
evidence_folder.mkdir(parents=True, exist_ok=True)

validation_results.to_csv(
    evidence_folder / "chronological_fold_results.csv",
    index=False
)
candidate_summary.to_csv(
    evidence_folder / "candidate_summary.csv",
    index=False
)
choose_comparison.to_csv(
    evidence_folder / "choose_tuned_comparison.csv",
    index=False
)

selected_record = pd.DataFrame([{
    "model": selected_candidate["model"],
    "settings": str(selected_candidate["settings"]),
    "selection_metric": "mean chronological validation Brier score",
    "selected_candidate_number": selected_candidate_number
}])

selected_record.to_csv(
    evidence_folder / "selected_settings.csv",
    index=False
)

print("Saved hyperparameter experiment evidence to:", evidence_folder)

Saved hyperparameter experiment evidence to: evidence\hyperparameter_experiments


## 12. Limitations and questions

### Limitations

- Only a small, manually declared set of settings is tested.
- The folds are chronological, but the historical dataset has already been explored in earlier work. This is a retrospective experiment, not a historically blind test.
- Choose is evaluated after settings are selected from Train folds. Do not repeatedly tune against Choose.
- The selected model still needs probability calibration and uncertainty analysis.
- Feature availability at the original booking timestamp remains unverified.
- No physical capacity or actual walk outcomes are present.

### Questions

1. Why is random cross-validation inappropriate for this time-ordered task?
2. Why do we split on unique arrival dates rather than arbitrary rows?
3. What does `C` control in logistic regression?
4. What is the effect of increasing `min_samples_leaf` in a random forest?
5. Why is the selected hyperparameter configuration not automatically the final production model?

**Next notebook:** `09_probability_calibration.ipynb`, where we check whether predicted probabilities match observed cancellation frequencies.